# Colab smoke test — upload and Run all

**1. Select a GPU runtime. 2. Runtime > Run all. 3. Keep the downloaded report ZIP.**

This version intentionally enables setup and the smoke test, as requested. Full training is **disabled**. It installs the pinned legacy environment separately from Colab's kernel, without a kernel restart or Drive authentication. It runs ten retained updates each of SGD and SAM on the authors' WRN-28-10 with CIFAR-10, batch 256 and eight logical sub-batches of 32. Diagnostic probes are additional and discarded; no final research result is claimed.

Checks cover finite losses, parameter updates, per-lane first-pass BatchNorm state, evaluation, checkpoints at updates 5 and 10, continuation after restoration, and evaluation after restoration. A final **PASS** requires both methods to pass. Setup/runtime failures produce a **FAILED** report and do not launch full training. Availability of legacy binaries and runtime GPU compatibility remain **(to verify)**. Nothing was executed during notebook preparation.

The smoke workers together have a one-hour wall-time limit after setup, within the cumulative 40-hour ledger. Files live in ephemeral `/content`; the final cell automatically downloads logs, results and the authoritative ledger. Save that ZIP before disconnecting. On a subsequent fresh runtime, upload the previous `authoritative_gpu_budget.json` to `/content` before Run all. The ledger must also accompany any move to another machine. Idle GPU allocation and environment installation are outside worker accounting.

**Observed T4 result (2026-11-16 local / November 17 UTC):** SGD completed all ten updates, evaluation and checkpoint restoration. SAM ran out of GPU memory at the eight-replica step, including after discarded diagnostic buffers were released. Both original-step numerical comparisons were inconclusive because the reference itself was not repeatable at the fixed tolerance. The combined smoke has NOT passed; use a larger GPU for further verification, preserving the ledger.


## 1. Automatic setup
No switches need changing for this smoke test. An isolated Python 3.8 environment retains JAX 0.2.6, CUDA jaxlib 0.1.57 and Flax 0.2.2. The authors' SAM code and model are unchanged. A standalone micromamba executable bootstraps the environment using its [official installation endpoint](https://mamba.readthedocs.io/en/latest/installation/micromamba-installation.html); its version is recorded. This does not replace Colab's Python or silently upgrade the research stack.

Setup and first compilation may take time. If dependency installation, GPU compatibility or memory fails, the report records it; the notebook never substitutes a smaller model, different optimizer or modern implementation. Passing this smoke test is required evidence before deciding on longer Colab runs; it does not guarantee that the full experiment fits Colab limits.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, subprocess, shutil, uuid, ast, signal, time

WORK_ROOT = Path('/content/sam_runall_smoke')  # Exported automatically before disconnecting.
ENV_PREFIX = WORK_ROOT / 'envs' / 'sam'
PYTHON = ENV_PREFIX / 'bin' / 'python'
REPO = Path('/content/sam_paper_smoke/work/sam')  # Pinned source; runtime outputs remain on persistent storage.
REVISION = 'f7b9ff46ee0deab8004d5508baf68b7e7c3dad1a'
DATA_DIR = WORK_ROOT / 'tensorflow_datasets'  # Or point to a complete existing TFDS cache.
REUSE_SESSION = None  # Absolute path to the original session when explicitly resuming.
START_NEW_SESSION = False  # Set True once to intentionally start a new experiment.
if REUSE_SESSION:
    SESSION = Path(REUSE_SESSION)
elif START_NEW_SESSION or 'SESSION' not in globals() or globals().get('SMOKE_NOTEBOOK_REVISION')!='diagnostic-memory-v5':
    SESSION = WORK_ROOT / 'smoke_runs' / (datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
SMOKE_NOTEBOOK_REVISION = 'diagnostic-memory-v5'
SEEDS = [0, 1, 2]
RUN_MOUNT_DRIVE = False
RUN_SMOKE_TEST = True
EXPORT_SMOKE = True
RUN_SETUP = True
RUN_PREFLIGHT = False
RUN_TRAINING = False
RENDER_RESULTS = False
REPLICATION_BUDGET_HOURS = 40.0  # Fixed cumulative ceiling, including preflight and retries.
BUDGET_FILE = WORK_ROOT / 'gpu_budget.json'  # Shared across sessions; never reset to regain time.
RUN_SELECTION = 'SAM-200_seed_0'  # One run, or explicitly 'all' for the nine-run matrix.
RESUME_EXISTING = False
CONFIG = dict(mode='smoke',budget_limit_hours=40.0,budget_file=str(BUDGET_FILE),revision=REVISION, repo=str(REPO), session=str(SESSION), data_dir=str(DATA_DIR),
              seeds=SEEDS, logical_replicas=8, batch_size=256, subbatch_size=32,
              lr=0.1, weight_decay=0.0005, rho=0.05, gradient_clipping=5.0)
print('Planned session:', SESSION)
SETUP_OK = False
FAILURES = []
SMOKE_DEADLINE = None
print('Run-all mode: setup + ten-batch smoke test enabled; full training disabled.')


In [ ]:
# No Drive mount and no kernel restart are required.
import traceback, urllib.request, tarfile, io, platform
WORK_ROOT.mkdir(parents=True,exist_ok=True)
SESSION.mkdir(parents=True,exist_ok=True)
prior_ledger=Path('/content/authoritative_gpu_budget.json')
if prior_ledger.exists():
    prior=json.loads(prior_ledger.read_text())
    assert prior['schema']==1 and prior['limit_seconds']==40*3600
    assert all(0<=a['charged_seconds']<=a['reserved_seconds'] for a in prior['attempts'])
    assert sum(a['charged_seconds'] for a in prior['attempts'])<=40*3600
    if BUDGET_FILE.exists():
        current=json.loads(BUDGET_FILE.read_text())
        known={a['id']:a for a in current['attempts']}
        assert all(known.get(a['id'])==a for a in prior['attempts']), 'Different ledger histories: reconcile before running.'
    else:
        shutil.copy2(prior_ledger,BUDGET_FILE)
print('Outputs:',SESSION)


In [ ]:
def setup_command(command):
    print('Setup:', ' '.join(map(str,command[:6])),flush=True)
    with (SESSION/'setup.log').open('a') as log:
        log.write('\nCOMMAND: '+repr(command)+'\n');log.flush()
        subprocess.run(list(map(str,command)),stdout=log,stderr=subprocess.STDOUT,check=True,timeout=1800)
    print('\n'.join((SESSION/'setup.log').read_text(errors='replace').splitlines()[-5:]),flush=True)

try:
    assert RUN_SETUP and not RUN_TRAINING
    assert sys.platform=='linux' and platform.machine()=='x86_64', 'Use a Linux x86-64 Colab GPU runtime.'
    setup_command(['nvidia-smi','-L'])
    manager=WORK_ROOT/'bin'/'micromamba'
    if not manager.exists():
        print('Downloading standalone environment manager (no kernel restart).',flush=True)
        with urllib.request.urlopen('https://micro.mamba.pm/api/micromamba/linux-64/latest',timeout=120) as response:
            data=response.read()
        with tarfile.open(fileobj=io.BytesIO(data),mode='r:bz2') as archive:
            member=archive.getmember('bin/micromamba')
            assert member.isfile()
            manager.parent.mkdir(parents=True,exist_ok=True)
            with archive.extractfile(member) as executable:
                manager.write_bytes(executable.read())
        manager.chmod(0o755)
        (SESSION/'bootstrap_sha256.txt').write_text(hashlib.sha256(manager.read_bytes()).hexdigest())
    setup_command([manager,'--version'])
    os.environ['MAMBA_ROOT_PREFIX']=str(WORK_ROOT/'mamba')
    REPO.parent.mkdir(parents=True,exist_ok=True)
    if not REPO.exists():
        setup_command(['git','clone','https://github.com/Siddharth-magesh/sam-replication.git',REPO])
        setup_command(['git','-C',REPO,'checkout','--detach',REVISION])
    assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==REVISION
    action='install' if PYTHON.exists() else 'create'
    setup_command([manager,action,'-p',ENV_PREFIX,'-c','conda-forge','--override-channels',
                   'python=3.8','pip=24.0','cudatoolkit=11.0','cudnn=8.0.5','-y'])
    requirements=[s.replace('absl-py==0.11.0','absl-py==0.10.0') for s in
                  (REPO/'requirements.txt').read_text().splitlines() if not s.startswith('jaxlib')]
    fixed=WORK_ROOT/'requirements_runtime.txt'
    fixed.write_text('\n'.join(requirements)+'\n')
    setup_command([PYTHON,'-m','pip','install','jaxlib==0.1.57+cuda110','-f',
                   'https://storage.googleapis.com/jax-releases/jax_cuda_releases.html'])
    setup_command([PYTHON,'-m','pip','install','-r',fixed])
    candidates=list(ENV_PREFIX.rglob('libdevice*.bc'))
    assert candidates,'Pinned CUDA libdevice missing.'
    libdir=WORK_ROOT/'cuda'/'nvvm'/'libdevice';libdir.mkdir(parents=True,exist_ok=True)
    shutil.copy2(candidates[0],libdir/'libdevice.10.bc')
    setup_command([PYTHON,'-c','from importlib.metadata import version; print({p:version(p) for p in ["numpy","tensorflow","jax","jaxlib","flax"]})'])
    SETUP_OK=True
    print('Setup complete. Proceeding automatically to smoke checks.')
except Exception as error:
    FAILURES.append(dict(stage='setup',error=str(error),traceback=traceback.format_exc()))
    print('SETUP FAILED; training will be skipped. Continue to the report cell for the diagnostic ZIP.')
    print(str(error))


## 2. Smoke test
The unchanged shared worker imports the original SAM training step. The smoke supervisor adds a one-hour timeout across both method launches. All output is logged. Failures are captured so Run all can still reach the report-download cell. Training, BatchNorm and checkpoint results remain **(to verify)** until this notebook finishes successfully on your runtime.


In [ ]:
WORKER_SOURCE = r'''# Orchestration around the authors' unchanged SAM implementation (Apache-2.0).
import os, sys, json, time, hashlib, traceback, subprocess, platform, shutil
from pathlib import Path
from functools import partial
phase, config_path, label, seed_text, resume_text = sys.argv[1:]
C = json.loads(Path(config_path).read_text())
seed, resume = int(seed_text), resume_text == 'true'
EPOCHS = {'SAM-200':200, 'SGD-200':200, 'SGD-400':400}
assert phase in ('preflight','train','smoke') and label in EPOCHS
assert C['mode'] in ('replication','smoke')
assert (phase=='smoke') == (C['mode']=='smoke')
assert phase!='smoke' or (label in ('SAM-200','SGD-200') and seed==0)
# A worker must be started by the persistent budget supervisor.
if 'SAM_STOP_MONOTONIC' not in os.environ:
    raise RuntimeError('Launch through the budget supervisor; direct worker execution is disabled.')
def budget_check():
    if time.monotonic() >= float(os.environ['SAM_STOP_MONOTONIC']):
        raise TimeoutError('Cumulative GPU budget reached; last committed checkpoint is preserved.')
budget_check()
run = Path(C['session'])/phase/(label+'_seed_'+str(seed))
run.mkdir(parents=True, exist_ok=True)
started, prior_seconds = time.monotonic(), 0.0
active_run = False

def write_json(path, value):
    path = Path(path); temporary = path.with_name(path.name+'.tmp')
    with temporary.open('w') as stream:
        stream.write(json.dumps(value, indent=2, allow_nan=False)+'\n'); stream.flush(); os.fsync(stream.fileno())
    os.replace(str(temporary), str(path))

def check(condition, message):
    if not condition:
        raise RuntimeError(message)

def run_experiment():
    global prior_seconds, active_run
    import numpy as np
    import tensorflow as tf
    tf.config.experimental.set_visible_devices([], 'GPU')
    import jax, jaxlib, flax
    import jax.numpy as jnp
    from flax import serialization
    from absl import flags
    sys.path.insert(0, str(Path(C['repo']).parent))
    from sam.sam_jax.models import load_model, wide_resnet
    from sam.sam_jax.datasets import dataset_source
    from sam.sam_jax.training_utils import flax_training as official
    FLAGS = flags.FLAGS
    FLAGS(['replication'])
    settings = dict(learning_rate=0.1, weight_decay=0.0005, gradient_clipping=5.0,
        use_rmsprop=False, sam_rho=0.05 if label=='SAM-200' else 0.0,
        sync_perturbations=False, label_smoothing=0.0, no_weight_decay_on_bn=False,
        ema_decay=0.0, use_test_set=True, use_additional_skip_connections=False,
        run_seed=seed, compute_top_5_error_rate=False)
    for key,value in settings.items():
        setattr(FLAGS,key,value)
    versions = dict(python=platform.python_version(), jax=jax.__version__,
        jaxlib=jaxlib.__version__, flax=flax.__version__, tensorflow=tf.__version__, numpy=np.__version__)
    check(sys.version_info[:2]==(3,8), 'Use the pinned Python 3.8 interpreter.')
    check(versions['jax']=='0.2.6' and versions['flax']=='0.2.2', 'JAX/Flax mismatch.')
    check(versions['jaxlib'].split('+')[0]=='0.1.57', 'Expected jaxlib 0.1.57; CUDA execution is checked next.')
    devices = jax.local_devices()
    check(len(devices)==1 and devices[0].platform=='gpu', 'Expose exactly one GPU.')
    gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name,uuid,memory.total','--format=csv,noheader'],text=True).strip()
    # Query the CUDA-visible ordinal, not all GPUs reported by nvidia-smi.
    import ctypes
    cuda = ctypes.CDLL('libcuda.so.1')
    ordinal = ctypes.c_int(); name = ctypes.create_string_buffer(256)
    check(cuda.cuInit(0)==0 and cuda.cuDeviceGet(ctypes.byref(ordinal),0)==0, 'CUDA identity query failed.')
    check(cuda.cuDeviceGetName(name,256,ordinal)==0, 'CUDA device name unavailable.')
    selected_gpu = name.value.decode()
    if C['mode']=='replication':
        check('A100' in selected_gpu, 'Main replication requires the selected GPU to be an A100.')
    check(versions['tensorflow']=='2.3.1' and versions['numpy']=='1.18.5', 'Legacy TensorFlow/NumPy pins changed.')
    revision = subprocess.check_output(['git','-C',C['repo'],'rev-parse','HEAD'],text=True).strip()
    check(revision==C['revision'], 'Repository revision mismatch.')
    check(not subprocess.check_output(['git','-C',C['repo'],'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Tracked runtime source was modified.')
    source_hashes = {str(p.relative_to(C['repo'])):hashlib.sha256(p.read_bytes()).hexdigest()
        for p in sorted(Path(C['repo']).rglob('*.py')) if '.git' not in p.parts}
    check(C['seeds']==[0,1,2] and C['batch_size']==256 and C['subbatch_size']==32 and C['logical_replicas']==8
          and C['lr']==0.1 and C['weight_decay']==0.0005 and C['rho']==0.05 and C['gradient_clipping']==5.0,
          'This notebook implements the fixed paper contract; config changes require a reviewed new experiment.')
    identity = dict(config=C, label=label, seed=seed, settings=settings, versions=versions,
        source_hashes=source_hashes, worker_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())
    signature = hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()
    if phase == 'train':
        for method in ('SAM-200','SGD-200'):
            pf = Path(C['session'])/'preflight'/(method+'_seed_0')
            report = json.loads((pf/'preflight.json').read_text())
            prior_manifest = json.loads((pf/'manifest.json').read_text())
            check(report['status']=='passed' and json.loads((pf/'status.json').read_text())['status']=='complete', 'Preflight is incomplete.')
            check(report['signature']==prior_manifest['signature'], 'Preflight provenance mismatch.')
            for key in ('config','versions','source_hashes','worker_sha256'):
                check(prior_manifest['identity'][key]==identity[key], 'Environment/source/config changed since preflight: '+key)
    manifest_path = run/'manifest.json'
    if manifest_path.exists():
        check(json.loads(manifest_path.read_text())['signature']==signature, 'Code/config/environment changed: use a new session.')
        old = json.loads((run/'status.json').read_text()) if (run/'status.json').exists() else {}
        prior_seconds = float(old.get('allocated_wall_seconds',0))
        if old.get('status')=='complete':
            print('Already complete:',run,flush=True)
            return
        check(resume and phase=='train', 'Incomplete existing run: explicitly resume or use a new session.')
    else:
        check(not resume, 'Cannot resume a run without a manifest.')
        write_json(manifest_path,dict(signature=signature,identity=identity))
    active_run = True
    write_json(run/'environment.json',dict(versions=versions,gpu=gpu,selected_gpu=selected_gpu,revision=revision,logical_replicas=8,sam_subbatch=32))
    write_json(run/'status.json',dict(status='running',allocated_wall_seconds=prior_seconds))
    np.random.seed(seed); tf.random.set_seed(seed)
    ds = dataset_source.Cifar10(256,'basic','none')
    check(ds.num_training_obs==50000, 'Expected 50,000 training examples.')
    module = wide_resnet.WideResnet.partial(blocks_per_group=4,channel_multiplier=10,num_outputs=10)
    # Explicit seed bypasses get_model's fixed default and PRNG-array truth test.
    model, initial_state = load_model.create_image_model(jax.random.PRNGKey(seed),32,32,module,3)
    # Independent factory construction is for architecture checking only.
    # Use its default key to avoid the legacy factory's PRNG-array truth test;
    # the actual training model above retains its unchanged explicit run seed.
    reference_model, reference_state = load_model.get_model(
        'WideResnet28x10', batch_size=32, image_size=32, num_classes=10, num_channels=3)
    parameter_count = sum(int(x.size) for x in jax.tree_leaves(model.params))
    reference_count = sum(int(x.size) for x in jax.tree_leaves(reference_model.params))
    assert parameter_count == reference_count, 'WRN-28-10 parameter count differs from official factory.'
    write_json(run/'architecture_check.json', dict(parameters=parameter_count,
        official_factory_parameters=reference_count, matched=True,
        blocks_per_group=4, channel_multiplier=10, num_outputs=10))
    del reference_model, reference_state
    optimizer = official.create_optimizer(model,0.0)
    initialization = dict(seed=seed,
        parameter_sha256=hashlib.sha256(serialization.to_bytes(model.params)).hexdigest(),
        parameters=sum(int(x.size) for x in jax.tree_leaves(model.params)))
    if (run/'initialization.json').exists():
        check(json.loads((run/'initialization.json').read_text()) == initialization,
              'Seeded initialization changed since this run started.')
    else:
        write_json(run/'initialization.json', initialization)
    states = jax.tree_map(lambda x:jnp.stack([x]*8),initial_state)
    schedule = official.get_cosine_schedule(EPOCHS[label],0.1,50000,256)
    original_step = partial(official.train_step,learning_rate_fn=schedule,l2_reg=0.0005)
    # Original pmean averages independent gradients BEFORE clipping and momentum.
    # pmean makes optimizer and LR unmapped. None avoids eight full optimizer copies.
    mapped_step = jax.jit(jax.vmap(original_step,in_axes=(None,0,0,0),
                                  out_axes=(None,0,0,None),axis_name='batch'))
    mapped_eval = jax.jit(jax.vmap(official.eval_step,in_axes=(None,0,0),axis_name='batch'))

    def synchronize(tree):
        for x in jax.tree_leaves(tree):
            if hasattr(x,'block_until_ready'):
                x.block_until_ready()
        return tree

    def finite(tree):
        return all(np.isfinite(np.asarray(x)).all() for x in jax.tree_leaves(tree))

    def keys_for(step,count):
        return jax.random.split(jax.random.fold_in(jax.random.PRNGKey(seed),int(step)),count)

    def train_batches(epoch):
        # Stateful TF augmentation/GPU kernels need not be bitwise deterministic.
        tf.random.set_seed(seed*100000+epoch)
        options = tf.data.Options(); options.experimental_deterministic=True
        return ds.get_train(True).with_options(options)

    def shard(batch,count=8):
        return {k:np.asarray(v).reshape((count,-1)+tuple(v.shape[1:])) for k,v in batch.items()}

    def differences(a,b):
        check(jax.tree_structure(a)==jax.tree_structure(b), 'Compared state structures differ.')
        max_abs,sqdiff,sqref = 0.0,0.0,0.0
        for x,y in zip(jax.tree_leaves(a),jax.tree_leaves(b)):
            check(np.shape(x)==np.shape(y), 'Compared array shapes differ.')
            x,y = np.asarray(x,dtype=np.float64),np.asarray(y,dtype=np.float64)
            max_abs=max(max_abs,float(np.max(np.abs(x-y))))
            sqdiff+=float(np.sum((x-y)**2)); sqref+=float(np.sum(y**2))
        return dict(max_abs=max_abs,relative_l2=(sqdiff**0.5)/max(sqref**0.5,1e-12))

    def eval_full(opt,bn,limit_batches=None):
        sums=dict(error_rate=0.0,loss=0.0); seen=0
        for batch_index,batch in enumerate(ds.get_test()):
            if limit_batches is not None and batch_index>=limit_batches:
                break
            budget_check()
            raw={k:np.asarray(v) for k,v in batch.items()}
            n=len(raw['label']); padded=((n+7)//8)*8
            raw={k:np.pad(v,[(0,padded-n)]+[(0,0)]*(v.ndim-1)) for k,v in raw.items()}
            raw['mask']=np.concatenate([np.ones(n),np.zeros(padded-n)]).astype(np.float32)
            out=synchronize(mapped_eval(opt.target,bn,shard(raw)))
            check(finite(out),'Non-finite evaluation.')
            for key in sums:
                sums[key]+=float(np.asarray(out[key])[0])
            seen+=n
        check(seen==(10000 if limit_batches is None else limit_batches*32),'Evaluation sample count mismatch.')
        return dict(test_error_pct=100*sums['error_rate']/seen,
            test_accuracy_pct=100*(1-sums['error_rate']/seen),test_loss=sums['loss']/seen,test_examples=seen)

    history=[]; completed=0
    checkpoints=run/'checkpoints'; checkpoints.mkdir(exist_ok=True)

    def save_checkpoint(epoch,opt,bn,rows):
        folder=checkpoints/('epoch_%04d'%epoch)
        check(not folder.exists(),'Checkpoint exists; refusing overwrite.')
        temporary=checkpoints/('writing_%04d'%epoch); temporary.mkdir(exist_ok=True)
        payload=dict(optimizer=opt,states=bn,completed_epoch=int(epoch))
        with (temporary/'state.msgpack').open('wb') as stream:
            stream.write(serialization.to_bytes(payload)); stream.flush(); os.fsync(stream.fileno())
        write_json(temporary/'history.json',rows)
        write_json(temporary/'metadata.json',dict(signature=signature,completed_epoch=epoch,checkpoint_unit='step' if phase=='smoke' else 'epoch',
            state_sha256=hashlib.sha256((temporary/'state.msgpack').read_bytes()).hexdigest()))
        os.replace(str(temporary),str(folder))
        write_json(checkpoints/'latest.json',dict(folder=folder.name))
        for old in sorted(checkpoints.glob('epoch_*'))[:-2]:
            check(old.resolve().parent==checkpoints.resolve(),'Unsafe checkpoint cleanup.')
            shutil.rmtree(str(old))
        return folder

    def load_checkpoint():
        committed = sorted(checkpoints.glob('epoch_*'))
        check(bool(committed), 'No complete epoch checkpoint; start a fresh run/session.')
        # Recover even if killed after directory commit but before latest.json update.
        folder=committed[-1]
        check(folder.resolve().parent==checkpoints.resolve(),'Invalid checkpoint pointer.')
        meta=json.loads((folder/'metadata.json').read_text()); data=(folder/'state.msgpack').read_bytes()
        check(meta['signature']==signature and hashlib.sha256(data).hexdigest()==meta['state_sha256'],'Checkpoint provenance mismatch.')
        template=dict(optimizer=optimizer,states=states,completed_epoch=0)
        restored=serialization.from_bytes(template,data)
        rows=json.loads((folder/'history.json').read_text()); epoch=int(restored['completed_epoch'])
        check(epoch==meta['completed_epoch']==len(rows),'Checkpoint/history disagree.')
        check(int(restored['optimizer'].state.step)==(epoch if phase=='smoke' else epoch*195),'Checkpoint step mismatch.')
        return restored['optimizer'],restored['states'],epoch,rows

    if resume:
        optimizer,states,completed,history=load_checkpoint()
        write_json(run/'history.json',history)
    if phase in ('preflight','smoke'):
        batch=next(iter(train_batches(0)))
        small={k:np.asarray(v)[:32][None,...] for k,v in batch.items()}
        one_state=jax.tree_map(lambda x:jnp.asarray(x)[None,...],initial_state); one_keys=keys_for(0,1)
        vmapped_one=jax.jit(jax.vmap(original_step,in_axes=(None,0,0,0),out_axes=(None,0,0,None),axis_name='batch'))
        pmapped_one=jax.pmap(original_step,axis_name='batch')
        reference=synchronize(pmapped_one(jax.tree_map(lambda x:jnp.asarray(x)[None,...],optimizer),one_state,small,one_keys))
        candidate=synchronize(vmapped_one(optimizer,one_state,small,one_keys))
        reference_optimizer=jax.tree_map(lambda x:x[0],reference[0])
        numerical = dict(parameters=differences(candidate[0].target,reference_optimizer.target),
                         optimizer_state=differences(candidate[0].state,reference_optimizer.state),
                         batchnorm=differences(candidate[1],reference[1]))
        checks=dict(one_replica_finite=finite(candidate) and finite(reference),one_replica_difference=numerical)
        write_json(run/'diagnostics.json',checks)
        # Measure the original GPU kernel's repeatability before interpreting a
        # cross-transform difference. This does not change the training step.
        repeated=synchronize(pmapped_one(jax.tree_map(lambda x:jnp.asarray(x)[None,...],optimizer),one_state,small,one_keys))
        repeated_optimizer=jax.tree_map(lambda x:x[0],repeated[0])
        repeat_difference=dict(parameters=differences(repeated_optimizer.target,reference_optimizer.target),
                               optimizer_state=differences(repeated_optimizer.state,reference_optimizer.state),
                               batchnorm=differences(repeated[1],reference[1]))
        strict_close=lambda ds:all(d['max_abs']<=1e-3 and d['relative_l2']<=1e-3 for d in ds.values())
        reference_repeatable=strict_close(repeat_difference)
        cross_transform_close=strict_close(numerical)
        noise_bound=all(numerical[k][metric]<=max(1e-3,3*repeat_difference[k][metric])
                       for k in numerical for metric in ('max_abs','relative_l2'))
        checks.update(reference_repeat_difference=repeat_difference,
                      reference_repeatable=reference_repeatable,
                      cross_transform_close=cross_transform_close,
                      within_reference_noise_bound=noise_bound,
                      numerical_equivalence=('passed' if reference_repeatable and cross_transform_close else
                          'inconclusive_original_not_repeatable' if not reference_repeatable and noise_bound else 'failed'))
        write_json(run/'diagnostics.json',checks)
        print('One-replica numerical comparison:',checks['numerical_equivalence'],flush=True)
        # A functional smoke may continue under measured reference variability;
        # it does NOT certify equivalence or authorize replication. Preflight
        # retains the strict threshold, including original-kernel repeatability.
        acceptable=(reference_repeatable and cross_transform_close) or (
            phase=='smoke' and not reference_repeatable and noise_bound)
        check(checks['one_replica_finite'] and finite(repeated) and acceptable,
              'One-replica numerical verification failed; inspect diagnostics.')
        del repeated,repeated_optimizer,reference,reference_optimizer,candidate,vmapped_one,pmapped_one
        # Free discarded comparison buffers before the full eight-lane step.
        import gc
        gc.collect()
        print('Checking eight logical replicas and BatchNorm.',flush=True)
        out=synchronize(mapped_step(optimizer,states,shard(batch),keys_for(0,8)))
        checks['eight_replica_finite']=finite(out)
        checks['shared_optimizer_step']=int(out[0].state.step)==int(optimizer.state.step)+1
        checks['shared_optimizer_shape']=all(np.shape(x)==np.shape(y) for x,y in zip(jax.tree_leaves(out[0]),jax.tree_leaves(optimizer)))
        checks['parameters_changed']=differences(out[0].target,optimizer.target)['max_abs']>0
        # Each lane's retained BN state must equal its CLEAN first pass, including for SAM.
        def clean_bn_forward(model,state,images,key):
            with flax.nn.stateful(state) as updated:
                with flax.nn.stochastic(key):
                    model(images,train=True)
            return updated
        bn_forward=jax.jit(clean_bn_forward)
        checks['bn_lanes']=[]
        grouped=shard(batch); lane_keys=keys_for(0,8)
        for lane in range(8):
            budget_check()
            state_lane=jax.tree_map(lambda x:x[lane],states)
            expected=synchronize(bn_forward(optimizer.target,state_lane,grouped['image'][lane],lane_keys[lane]))
            actual=jax.tree_map(lambda x:x[lane],out[1])
            checks['bn_lanes'].append(differences(actual,expected))
        checks['bn_updated']=differences(out[1],states)['max_abs']>0
        checks['bn_lane_variation']=max(float(np.max(np.abs(np.asarray(x)-np.asarray(x)[0]))) for x in jax.tree_leaves(out[1]))
        write_json(run/'diagnostics.json',checks)
        check(checks['bn_updated'] and checks['bn_lane_variation']>0,'BN states did not update independently.')
        check(all(d['max_abs']<=2e-5 and d['relative_l2']<=2e-4 for d in checks['bn_lanes']), 'Retained BN differs from clean first pass.')
        checks['eight_bn_states']=all(x.shape[0]==8 for x in jax.tree_leaves(out[1]))
        raw_eval=next(iter(ds.get_test()))
        many_eval=synchronize(mapped_eval(optimizer.target,out[1],shard(raw_eval)))
        mean_bn=jax.tree_map(lambda x:jnp.mean(x,axis=0)[None,...],out[1])
        one_eval=jax.jit(jax.vmap(official.eval_step,in_axes=(None,0,0),axis_name='batch'))
        ref_eval=synchronize(one_eval(optimizer.target,mean_bn,shard(raw_eval,1)))
        checks['bn_eval_difference']=differences(jax.tree_map(lambda x:x[0],many_eval),jax.tree_map(lambda x:x[0],ref_eval))
        write_json(run/'diagnostics.json',checks)
        check(checks['eight_replica_finite'] and checks['eight_bn_states'] and checks['shared_optimizer_step'] and checks['shared_optimizer_shape'] and checks['parameters_changed'],
            'Logical-replica update check failed.')
        check(checks['bn_eval_difference']['relative_l2']<=1e-3,'BatchNorm evaluation check failed.')
        del out,many_eval,ref_eval,expected,actual
        # Restart timing from untouched initialization; keep the full 200-epoch schedule.
    if phase=='smoke':
        # Ten committed training updates; diagnostic probes above are discarded.
        smoke_rows=[]
        initial_parameters=optimizer.target
        for batch_index,batch in enumerate(train_batches(0)):
            if batch_index==10:
                break
            budget_check()
            output=synchronize(mapped_step(optimizer,states,shard(batch),keys_for(optimizer.state.step,8)))
            optimizer,states,metrics,lr=output
            check(finite(output),'Non-finite smoke update.')
            check(int(optimizer.state.step)==batch_index+1,'Smoke optimizer step mismatch.')
            row=dict(step=batch_index+1,loss=float(np.mean(np.asarray(metrics['train_loss']))),
                     train_error=float(np.mean(np.asarray(metrics['train_error_rate']))),lr=float(np.asarray(lr)))
            smoke_rows.append(row); print(json.dumps(row),flush=True)
            if batch_index+1 in (5,10):
                save_checkpoint(batch_index+1,optimizer,states,smoke_rows)
                ro,rs,re,rh=load_checkpoint()
                check(re==batch_index+1 and rh==smoke_rows,'Smoke checkpoint metadata mismatch.')
                check(differences(ro,optimizer)['max_abs']==0 and differences(rs,states)['max_abs']==0,'Smoke checkpoint restoration changed tensors.')
                # Continue from the restored optimizer + all eight BN states after step 5.
                optimizer,states=ro,rs
        check(len(smoke_rows)==10 and differences(optimizer.target,initial_parameters)['max_abs']>0,'Smoke training did not update parameters.')
        evaluated=eval_full(optimizer,states,limit_batches=10)
        ro,rs,re,rh=load_checkpoint()
        restored_eval=eval_full(ro,rs,limit_batches=10)
        check(abs(evaluated['test_loss']-restored_eval['test_loss'])<=1e-5 and evaluated['test_error_pct']==restored_eval['test_error_pct'], 'Restored evaluation differs.')
        write_json(run/'smoke.json',dict(status='passed',training_batches=10,training_examples=2560,
                   evaluation=evaluated,restored_evaluation=restored_eval,checkpoint_restored_at_steps=[5,10],
                   parameters_changed=True,finite_losses=True,history=smoke_rows,diagnostics=checks))
        write_json(run/'status.json',dict(status='complete',training_batches=10,resumed=False,
                   allocated_wall_seconds=prior_seconds+time.monotonic()-started))
        return
    startup_seconds=time.monotonic()-started
    epoch_limit=1 if phase=='preflight' else EPOCHS[label]
    for epoch in range(completed,epoch_limit):
        budget_check()
        start=time.monotonic(); after_first=None; count=0; step_seconds=[]; tick=start
        for batch in train_batches(epoch):
            budget_check()
            output=synchronize(mapped_step(optimizer,states,shard(batch),keys_for(optimizer.state.step,8)))
            optimizer,states,metrics,lr=output
            check(finite(metrics) and finite(lr),'Non-finite step metrics.')
            now=time.monotonic(); step_seconds.append(now-tick); tick=now
            count+=1
            if count==1:
                after_first=time.monotonic()
        train_seconds=time.monotonic()-start
        steady_epoch_seconds=(time.monotonic()-after_first)*195/194
        check(count==195,'Expected 195 full batches per epoch.')
        check(finite(optimizer) and finite(states),'Non-finite epoch state.')
        eval_start=time.monotonic(); row=eval_full(optimizer,states)
        row.update(epoch=epoch+1,method=label,seed=seed,last_lr=float(np.asarray(lr)),
            train_seconds=train_seconds,eval_seconds=time.monotonic()-eval_start,
            cumulative_train_seconds=sum(x['train_seconds'] for x in history)+train_seconds)
        history.append(row)
        checkpoint_start=time.monotonic(); save_checkpoint(epoch+1,optimizer,states,history)
        checkpoint_seconds=time.monotonic()-checkpoint_start
        write_json(run/'history.json',history); print(json.dumps(row),flush=True)
        if phase=='preflight':
            ro,rs,re,rh=load_checkpoint()
            check(re==1 and rh==history,'Checkpoint metadata round-trip failed.')
            check(differences(ro,optimizer)['max_abs']==0 and differences(rs,states)['max_abs']==0,'Checkpoint state round-trip failed.')
            write_json(run/'preflight.json',dict(status='passed',signature=signature,checkpoint_roundtrip=True,
                estimated_steady_epoch_seconds=steady_epoch_seconds,evaluation_seconds=row['eval_seconds'],
                checkpoint_seconds=checkpoint_seconds,first_step_and_input_seconds=after_first-start,
                startup_and_diagnostics_seconds=startup_seconds,measured_train_seconds=train_seconds,
                steady_step_mean_seconds=float(np.mean(step_seconds[1:])),
                steady_step_p90_seconds=float(np.percentile(step_seconds[1:],90)),step_seconds=step_seconds,
                planning_epoch_seconds=max(train_seconds,195*float(np.percentile(step_seconds[1:],90)))+row['eval_seconds']+checkpoint_seconds,
                note='Actual synchronized epoch, evaluation and checkpoint timings; p90 steps give conservative planning. Startup includes diagnostic compilation.'))
        write_json(run/'status.json',dict(status='running',completed_epochs=epoch+1,resumed=resume,
            allocated_wall_seconds=prior_seconds+time.monotonic()-started))
    write_json(run/'status.json',dict(status='complete',completed_epochs=epoch_limit,resumed=resume,
        allocated_wall_seconds=prior_seconds+time.monotonic()-started,
        allocated_gpu_hours=(prior_seconds+time.monotonic()-started)/3600))

try:
    run_experiment()
except BaseException as error:
    write_json(run/('failure_%d.json'%int(time.time())),dict(type=type(error).__name__,message=str(error),traceback=traceback.format_exc()))
    if active_run or not (run/'manifest.json').exists():
        write_json(run/'status.json',dict(status='failed',allocated_wall_seconds=prior_seconds+time.monotonic()-started))
    raise
'''
ast.parse(WORKER_SOURCE, feature_version=(3, 8))
print('Runner syntax parsed only; GPU behavior (to verify).')

SUPERVISOR_SOURCE = r'''# Linux-only wall-clock budget supervisor. Does not import any ML packages.
import os, sys, json, time, uuid, signal, subprocess, fcntl, ctypes
from pathlib import Path
config_path, phase, method, seed, resume = sys.argv[1:]
C=json.loads(Path(config_path).read_text())
LIMIT=40*3600.0
assert C['budget_limit_hours']==40.0
ledger_path=Path(C['budget_file']); ledger_path.parent.mkdir(parents=True,exist_ok=True)

def atomic(path,value):
    path=Path(path); tmp=path.with_name(path.name+'.tmp')
    with tmp.open('w') as f:
        json.dump(value,f,indent=2,allow_nan=False); f.write('\n'); f.flush(); os.fsync(f.fileno())
    os.replace(str(tmp),str(path))

def interrupt(signum,frame):
    raise KeyboardInterrupt('Supervisor interrupted')

signal.signal(signal.SIGTERM,interrupt)
with (ledger_path.with_suffix('.lock')).open('a+') as lock:
    # Concurrent launches fail; no stale lock-file deletion or automatic budget reset.
    fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    ledger=json.loads(ledger_path.read_text()) if ledger_path.exists() else dict(schema=1,limit_seconds=LIMIT,attempts=[])
    assert ledger['schema']==1 and ledger['limit_seconds']==LIMIT
    assert all(0<=e['charged_seconds']<=e['reserved_seconds'] for e in ledger['attempts'])
    used=sum(e['charged_seconds'] for e in ledger['attempts'])
    assert 0<=used<=LIMIT
    remaining=LIMIT-used
    if remaining<=30:
        raise RuntimeError('Cumulative 40-hour budget exhausted (30-second shutdown reserve).')
    start=time.monotonic()
    attempt_seconds=min(remaining,3600.0,float(os.environ.get('SAM_ATTEMPT_LIMIT_SECONDS','3600')))
    assert attempt_seconds>30, 'Smoke-test time limit reached.'
    deadline=start+attempt_seconds
    entry=dict(id=uuid.uuid4().hex,session=C['session'],phase=phase,method=method,seed=int(seed),
               started_unix=time.time(),state='reserved',reserved_seconds=remaining,charged_seconds=remaining)
    # Reserve ALL remaining time durably before spawning. A crashed supervisor never refunds it.
    ledger['attempts'].append(entry); atomic(ledger_path,ledger)
    process=None; code=1; budget_stopped=False; confirmed_exit=True
    parent_pid=os.getpid()
    def child_setup():
        os.setsid()
        libc=ctypes.CDLL(None)
        if libc.prctl(1,signal.SIGKILL)!=0 or os.getppid()!=parent_pid:
            os._exit(125)  # Linux parent-death signal: no orphan GPU worker.
    def stop_child():
        if process is None or process.poll() is not None:
            return
        try:
            os.killpg(process.pid,signal.SIGINT)
            process.wait(timeout=min(3,max(0.1,deadline-time.monotonic()-5)))
        except (subprocess.TimeoutExpired,ProcessLookupError):
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGKILL)
                process.wait(timeout=5)
    try:
        env=os.environ.copy()
        env['SAM_STOP_MONOTONIC']=str(deadline-20)
        process=subprocess.Popen([sys.executable,'-u',str(Path(C['session'])/'runner.py'),
                   phase,config_path,method,seed,resume],env=env,preexec_fn=child_setup)
        try:
            code=process.wait(timeout=max(0.1,deadline-time.monotonic()-10))
        except subprocess.TimeoutExpired:
            budget_stopped=True; code=124; stop_child()
        # Cooperative deadline stops may exit before the outer watchdog.
        if code!=0 and time.monotonic()>=deadline-21:
            budget_stopped=True
    except KeyboardInterrupt:
        code=130
    finally:
        # Ignore further interrupts while killing/reaping the child and settling the ledger.
        signal.signal(signal.SIGINT,signal.SIG_IGN); signal.signal(signal.SIGTERM,signal.SIG_IGN)
        try:
            stop_child()
        except BaseException:
            confirmed_exit=False
        elapsed=time.monotonic()-start
        if confirmed_exit and (process is None or process.poll() is not None):
            entry.update(state='budget_stopped' if budget_stopped else 'finished',
                         charged_seconds=min(remaining,elapsed),observed_seconds=elapsed,exit_code=code,
                         finished_unix=time.time())
        else:
            entry.update(state='unresolved',exit_code=code,observed_seconds=elapsed)
            # Keep the entire reservation charged. No further launch is possible.
        atomic(ledger_path,ledger)
        run=Path(C['session'])/phase/(method+'_seed_'+seed)
        run.mkdir(parents=True,exist_ok=True)
        atomic(run/('budget_attempt_'+entry['id']+'.json'),entry)
        if budget_stopped:
            path=run/'status.json'
            status=json.loads(path.read_text()) if path.exists() else {}
            status.update(status='budget_stopped',budget_attempt=entry['id'])
            atomic(path,status)
        print('Cumulative charged GPU hours:',sum(e['charged_seconds'] for e in ledger['attempts'])/3600,
              '/ 40; attempt:',entry['state'],flush=True)
        if elapsed>remaining or not confirmed_exit:
            print('Shutdown/accounting anomaly: budget locked; inspect scheduler accounting.',flush=True)
            code=124
    sys.exit(code)
'''
ast.parse(SUPERVISOR_SOURCE, feature_version=(3, 8))


In [ ]:
def budget_snapshot():
    assert REPLICATION_BUDGET_HOURS==40.0 and CONFIG['budget_limit_hours']==40.0
    if BUDGET_FILE.exists():
        data=json.loads(BUDGET_FILE.read_text())
        assert data['schema']==1 and data['limit_seconds']==40*3600
        assert all(0<=a['charged_seconds']<=a['reserved_seconds'] for a in data['attempts'])
        charged=sum(a['charged_seconds'] for a in data['attempts'])
        assert 0<=charged<=40*3600
    else:
        charged=0.0
    return dict(charged_hours=charged/3600,remaining_hours=40-charged/3600)

def prepare_session():
    assert PYTHON.is_file() and REPO.is_dir(), 'Complete setup or configure the pinned environment.'
    assert SESSION.resolve()!=REPO.resolve() and REPO.resolve() not in SESSION.resolve().parents
    SESSION.mkdir(parents=True,exist_ok=True)
    files=[('config.json',json.dumps(CONFIG,indent=2)+'\n'),('runner.py',WORKER_SOURCE),('supervisor.py',SUPERVISOR_SOURCE)]
    for name,text in files:
        path=SESSION/name
        if path.exists():
            assert path.read_text()==text, 'Session code/config changed: use a new session, keeping the SAME budget ledger.'
        else:
            path.write_text(text)
    return SESSION/'supervisor.py'

def launch(phase,label,seed,resume=False):
    supervisor=prepare_session()
    folder=SESSION/phase/(label+'_seed_'+str(seed))
    if (folder/'status.json').exists() and json.loads((folder/'status.json').read_text()).get('status')=='complete':
        manifest=json.loads((folder/'manifest.json').read_text())
        assert manifest['identity']['config']==CONFIG
        assert manifest['identity']['worker_sha256']==hashlib.sha256(WORKER_SOURCE.encode()).hexdigest()
        print('Already complete; no launch:',folder)
        return
    assert budget_snapshot()['remaining_hours']>30/3600,'Cumulative budget exhausted.'
    environment=os.environ.copy()
    environment['TFDS_DATA_DIR']=str(DATA_DIR)
    environment['SAM_ATTEMPT_LIMIT_SECONDS']=str(max(0,SMOKE_DEADLINE-time.monotonic()))
    environment['PYTHONPATH']=str(REPO.parent)
    environment['PYTHONHASHSEED']=str(seed)
    environment['LD_LIBRARY_PATH']=str(ENV_PREFIX/'lib')+':'+environment.get('LD_LIBRARY_PATH','')
    environment['XLA_PYTHON_CLIENT_PREALLOCATE']='false'
    environment.setdefault('CUDA_VISIBLE_DEVICES','0')
    if (WORK_ROOT/'cuda'/'nvvm'/'libdevice'/'libdevice.10.bc').exists():
        environment['XLA_FLAGS']='--xla_gpu_cuda_data_dir='+str(WORK_ROOT/'cuda')
    log=SESSION/(phase+'_'+label+'_seed_'+str(seed)+'.log')
    # Regular-file stdout lets the independent supervisor survive notebook disconnection.
    with log.open('a') as stream,log.open('r') as reader:
        reader.seek(0,2)
        process=subprocess.Popen([str(PYTHON),'-u',str(supervisor),str(SESSION/'config.json'),
                    phase,label,str(seed),str(resume).lower()],cwd=str(REPO.parent),env=environment,
                    stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)
        try:
            while process.poll() is None:
                print(reader.read(),end=''); time.sleep(1)
            print(reader.read(),end='')
        except KeyboardInterrupt:
            # Do not kill the supervisor abruptly: it must reap the worker and settle accounting.
            process.send_signal(signal.SIGINT)
            process.wait()
            raise
    print('Persistent budget:',budget_snapshot())
    assert process.returncode==0,'Worker stopped: inspect '+str(log)+'. Last committed checkpoint remains available.'

if RUN_SMOKE_TEST and SETUP_OK:
    SMOKE_DEADLINE=time.monotonic()+3600
    for method in ['SGD-200','SAM-200']:
        try:
            assert time.monotonic()<SMOKE_DEADLINE-30, 'One-hour smoke limit reached.'
            launch('smoke',method,0)
        except Exception as error:
            FAILURES.append(dict(stage=method,error=str(error),traceback=traceback.format_exc()))
            print('SMOKE FAILED:',method,str(error))
            break
else:
    print('Smoke execution skipped because setup failed or the smoke switch is off.')


## 3. Final result and automatic download
**PASS** means the functional checks passed: ten updates per method, evaluation and checkpoint restoration. Numerical equivalence is reported separately. If the original GPU step is not repeatable at the fixed tolerance, differences within three times its measured variability allow only the functional smoke to continue; this is not proof of equivalence. The main A100 preflight retains its strict numerical gate. **FAILED / INCOMPLETE** means do not start the long experiment yet. Send the downloaded ZIP for review in either case. The ZIP includes the latest authoritative ledger; save it before this ephemeral runtime disappears. Browser download permission may be required. If the kernel or VM crashes completely, rerun only this report cell while its files remain available.


In [ ]:
reports=[]
for method in ['SGD-200','SAM-200']:
    path=SESSION/'smoke'/(method+'_seed_0')/'smoke.json'
    if path.exists():
        try:
            manifest=json.loads((path.parent/'manifest.json').read_text())
            assert manifest['identity']['config']==CONFIG
            assert manifest['identity']['worker_sha256']==hashlib.sha256(WORKER_SOURCE.encode()).hexdigest()
            assert json.loads((path.parent/'status.json').read_text())['status']=='complete'
            r=json.loads(path.read_text())
            assert r['status']=='passed' and r['training_batches']==10 and r['evaluation']['test_examples']==320
            reports.append(dict(method=method,status='passed',batches=10,finite_losses=r['finite_losses'],
                       parameters_changed=r['parameters_changed'],restored_at=r['checkpoint_restored_at_steps'],
                       numerical_equivalence=r['diagnostics']['numerical_equivalence']))
        except Exception as error:
            FAILURES.append(dict(stage='report_'+method,error=str(error)))
passed=SETUP_OK and len(reports)==2 and not FAILURES
summary=dict(status='PASS' if passed else 'FAILED / INCOMPLETE',reports=reports,failures=FAILURES,
             full_training_launched=False,replication_ready=False,session=str(SESSION),
             numerical_note='Functional smoke only. Inconclusive numerical equivalence requires investigation; strict A100 preflight remains required.')
(SESSION/'smoke_summary.json').write_text(json.dumps(summary,indent=2))
print('\nFINAL RESULT:',summary['status'])
print(json.dumps(reports,indent=2))
if FAILURES:
    print('Do not start the full experiment. Share the diagnostic ZIP.')
if EXPORT_SMOKE:
    import zipfile
    from google.colab import files
    archive=SESSION/'smoke_reports.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as bundle:
        for path in SESSION.rglob('*'):
            if path.is_file() and path.suffix in ('.json','.log','.txt') and 'checkpoints' not in path.parts:
                bundle.write(path,str(path.relative_to(SESSION)))
        if BUDGET_FILE.exists():
            bundle.write(BUDGET_FILE,'authoritative_gpu_budget.json')
    print('Report archive:',archive)
    files.download(str(archive))
